# CH-2 耗電預測與出水溫度調高節電評估

1. 用**冷凍噸、出水溫度**預測 CH-2（額定 400 RT）的耗電，不用氣象資料
2. 評估出水溫度調整 **−1、−0.5、+0.5、+1、+1.5°C** 時的節電量（冷負載不變）；負荷 > 360 RT 或剛開機的小時不調整

## 報告摘要

資料期間 2026/09/17 ~ 10/13（約 4 週），只算主機耗電，不含泵浦與冷卻水塔。

**重點**
- 調高冰水出水溫度會省電：調高 0.5~1°C，主機節電約 **0.6%~2.8%**
- 出水溫度過去幾乎沒調過，只能給範圍（資料推估的 95% 區間下限略低於 0），**建議試行 2~4 週以實測確認**

| 出水溫度調整 | 保守估計（實際資料） | 理論估計（上限） |
|---|---|---|
| 調高 0.5°C | 節電 0.6% | 節電 1.4% |
| 調高 1.0°C | 節電 1.1% | 節電 2.8% |
| 調高 1.5°C（僅供參考） | 節電 1.7% | 節電 4.2% |
| 調低 0.5°C | 耗電增加 0.6% | 耗電增加 1.4% |
| 調低 1.0°C（僅供參考） | 耗電增加 1.1% | 耗電增加 2.8% |

「僅供參考」超出過去出水溫度範圍（約 8.5~9.8°C）。

**怎麼估的**
- 耗電模型：冷凍噸 + 出水溫度的逐時模型，與電錶實測差約 3.1%（ASHRAE Guideline 14 要求 30% 以內）
- 節電量：同一小時用實際與調整後的出水溫度各預測一次相減，模型誤差互相抵銷
- 保守 = 實際資料推估；理論 = 卡諾循環上限。實際落在哪一端要靠試行確認
- 調高、調低效果幾乎一樣大：主機要克服約 27°C 的溫度落差，調 1°C 只改變不到 4%

**試行建議**：選負載相近的工作日，出水溫度輪流設定（例如 9.0°C 與 10.0°C），各 1~2 週；記錄主機耗電、冷凍噸、室內溫濕度。

**限制**：假設冰水溫差、流量不變；只有 9~10 月資料；不含泵浦與冷卻水塔。

## 分析方法

- **模型**：灰箱線性迴歸（參考 DOE-2），耗電 = f(冷凍噸、冷凍噸²、出水溫度、出水溫度 × 冷凍噸)
- **訓練資料**：CH-2 穩定運轉逐時資料，排除冷凍噸 > 360 RT，共 72 小時、15 天
- **準確度**：依天分組交叉驗證 CV(RMSE) 約 3.1%
- **節電量**：同一小時「預測（實際出水）− 預測（調整後出水）」；節電率 = 節電量 ÷ 實測耗電

| 決定 | 理由 |
|---|---|
| 用灰箱線性 | CV(RMSE) 3.1%，遠低於 LightGBM（12~13%）；出水調高時耗電下降，方向正確 |
| 不用 LightGBM | 出水溫度高的時段負荷也高，未加限制的樹模型算出調高反而更耗電 |
| 不用氣象資料 | 案場拿不到天氣資料 |
| 卡諾循環只估上限 | 需要冷凝溫度，沒有量測 |
| 排除負荷 > 360 RT | 接近滿載時出水溫度是被負荷推高的，會把「負荷大」誤算成「出水溫度高」 |
| 剛開機的小時不調整 | 水溫還在下降，超出模型範圍 |
| 節電用「預測 − 預測」 | 用實測相減，模型誤差（可達數十 kW）會被當成節電 |

## 1. 環境設定

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

# 切到專案根目錄，才 import 得到 db、data、model
if Path.cwd().name == "analysis":
    os.chdir(Path.cwd().parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from research import ch2_model as ch2

pio.renderers.default = "jupyterlab"

DELTAS = [-1.0, -0.5, 0.5, 1.0, 1.5]  # 出水溫度調整情境 (°C)

### 圖表統一格式

In [2]:
# 顏色、名稱、單位、版面統一放在 research/chart_style.py
from functools import partial

from research import chart_style
from research.chart_style import REF_COLOR, date_axis, label

chart_style.NAMES["kW"] = "耗電"  # 座標軸寫成「耗電 (kW)」
show = partial(chart_style.show, legend="bottom")  # 這本圖例都放圖下方

## 2. 資料整理

- CH-2 每分鐘資料，取穩定運轉時間的每小時平均
- 冷凍噸 > 360 RT（約額定 90%）時主機接近滿載、出水被負荷推高，**不建模、情境中也不調整**

In [3]:
trainer = ch2.AnalysisTrainer()
await trainer.main()
minute_data = trainer.df_min
steady_data = trainer.df_steady
model_data = trainer.df

print(
    f"每分鐘資料 {len(minute_data)} 筆，"
    f"{minute_data.index.min():%m/%d} ~ {minute_data.index.max():%m/%d}"
)
summary_groups = [
    ("負荷過大", steady_data[steady_data["LIMITED"]]),
    ("建模資料", model_data),
]
for group_name, group_data in summary_groups:
    day_count = group_data.index.normalize().nunique()
    print(
        f"{group_name}：{len(group_data)} 小時（{day_count} 天），"
        f"平均冷凍噸 {group_data['Q_COOL'].mean():.0f} RT、"
        f"出水溫度 {group_data['T_CHWS'].mean():.2f}°C、"
        f"耗電 {group_data['kW'].mean():.0f} kW"
    )
model_data[["kW", "T_CHWS", "T_CHWR", "DT", "FLOW", "Q_COOL"]].describe().round(2)

每分鐘資料 37322 筆，09/17 ~ 10/13
負荷過大：25 小時（8 天），平均冷凍噸 381 RT、出水溫度 9.31°C、耗電 235 kW
建模資料：72 小時（15 天），平均冷凍噸 304 RT、出水溫度 8.82°C、耗電 173 kW


,kW,T_CHWS,T_CHWR,DT,FLOW,Q_COOL
count,72.00,72.00,72.00,72.00,72.00,72.00
mean,173.29,8.82,13.01,4.19,3677.66,303.80
std,32.51,0.27,0.73,0.55,352.51,35.40
min,103.50,8.47,11.85,3.15,2976.50,218.73
25%,158.61,8.63,12.41,3.83,3457.38,289.38
50%,177.80,8.70,12.97,4.32,3695.44,310.16
75%,195.22,8.97,13.60,4.65,3978.71,323.48
max,225.07,9.78,14.41,5.21,4238.78,359.95


## 3. 選模型

依天分組 5 折交叉驗證（標準：CV(RMSE) < 30%、NMBE 在 ±10% 內），並檢查出水 +0.5°C 時耗電是否下降。

| 模型 | 特徵 |
|---|---|
| LightGBM | 出水溫度、回水溫度、流量 |
| LightGBM（溫差） | 回水溫度改成溫差 |
| LightGBM（溫差 + 單調限制） | 限制出水溫度越高耗電越低 |
| 灰箱線性（無氣象） | 冷凍噸、冷凍噸²、出水溫度、出水溫度 × 冷凍噸 |

固定採用灰箱線性，其他列出比較。

In [4]:
# 方向檢查：出水 +1°C 時模型與卡諾理論的耗電變化
print("選出的模型：", trainer.model_name)
model_change = ch2.change_pct(trainer.model, model_data, 1.0, trainer.make_x)
ideal_change = ch2.ideal_change_pct(model_data, 1.0)
print(
    f"出水 +1°C 耗電變化（建模小時）：{trainer.model_name} {model_change:.2f}%，"
    f"卡諾理論 {ideal_change:.2f}%"
)
trainer.comparison.round(2)

選出的模型： 灰箱線性（無氣象）
出水 +1°C 耗電變化（建模小時）：灰箱線性（無氣象） -1.64%，卡諾理論 -4.02%


,CV(RMSE) %,NMBE %,MAE kW,R²,出水 +0.5°C 耗電變化 %,出水 -0.5°C 耗電變化 %
模型,,,,,,
LightGBM,13.08,0.39,18.80,0.51,5.03,-6.00
LightGBM（溫差）,12.57,-0.10,18.35,0.55,2.91,-0.49
LightGBM（溫差 + 單調限制）,12.40,-1.22,17.73,0.56,-0.78,1.04
灰箱線性（無氣象）,3.13,0.25,4.37,0.97,-0.82,0.82


In [5]:
# 出水溫度偏高的時段，負載也偏高（全部穩定運轉小時）
supply_load_corr = steady_data["T_CHWS"].corr(steady_data["Q_COOL"])
print("出水溫度與冷凍噸相關係數：", round(supply_load_corr, 2))
load_bins = pd.cut(steady_data["Q_COOL"], [0, 300, 360, 500])
(
    steady_data.groupby(load_bins, observed=True)["T_CHWS"]
    .agg(["count", "mean", "max"])
    .round(2)
    .rename_axis("冷凍噸區間 (RT)")
)

出水溫度與冷凍噸相關係數： 0.57


,count,mean,max
冷凍噸區間 (RT),,,
"(0, 300]",26,8.72,9.08
"(300, 360]",46,8.87,9.78
"(360, 500]",25,9.31,10.47


### 結論

- **LightGBM**：CV(RMSE) 12~13%，方向錯誤或幾乎沒反應，因為出水溫度高的時段多半負荷也高（上表）
- **灰箱線性**：CV(RMSE) 3.1%、NMBE 接近 0；建模小時出水每調高 1°C 耗電約 −1.6%，低於卡諾理論值（約 −4.0%）

## 4. CH-2 耗電模型

In [6]:
metric_texts = []
for metric_name, metric_value in trainer.metric.items():
    metric_texts.append(f"{metric_name} = {metric_value:.2f}")
print(trainer.model_name, "交叉驗證：" + "、".join(metric_texts))
trainer.coefficients().round(5).to_frame("係數")

灰箱線性（無氣象） 交叉驗證：CV(RMSE) % = 3.13、NMBE % = 0.25、MAE kW = 4.37、R² = 0.97


,係數
截距,-42.18738
Q,0.56276
Q2,0.00074
T_CHWS,-0.01261
Q_x_T_CHWS,-0.00929


In [7]:
# 預測 vs 實際：X、Y 軸同範圍，取整到 25 kW
TICK = 25
axis_values = model_data[["kW", "kW_CV"]]
axis_low = np.floor(axis_values.min().min() / TICK) * TICK
axis_high = np.ceil(axis_values.max().max() / TICK) * TICK

fig = go.Figure()
fig.add_trace(
    go.Scattergl(
        x=model_data["kW"],
        y=model_data["kW_CV"],
        mode="markers",
        name="逐時資料",
        marker=dict(
            size=8,
            color="#2a78d6",
            opacity=0.6,
            line=dict(width=1, color="white"),
        ),
        text=model_data.index.strftime("%m/%d %H:%M"),
        hovertemplate=(
            "%{text}<br>實際: %{x:.0f} kW<br>預測: %{y:.0f} kW<extra></extra>"
        ),
    )
)
fig.add_trace(
    go.Scatter(
        x=[axis_low, axis_high],
        y=[axis_low, axis_high],
        mode="lines",
        name="預測 = 實際",
        line=dict(color=REF_COLOR, width=1.5, dash="dash"),
    )
)
# 繪圖區縮成正方形，座標範圍不變
square_axis = dict(
    range=[axis_low, axis_high],
    tick0=axis_low,
    dtick=TICK,
    constrain="domain",
)
fig.update_xaxes(title_text="實際耗電 kW (kW)", **square_axis)
fig.update_yaxes(
    title_text="預測耗電 kW (kW)",
    scaleanchor="x",
    scaleratio=1,
    **square_axis,
)
metric = trainer.metric
show(
    fig,
    "CH-2 耗電：預測 vs 實際",
    f"依天交叉驗證（{len(model_data)} 小時）：CV(RMSE) {metric['CV(RMSE) %']:.1f}%、"
    f"NMBE {metric['NMBE %']:+.1f}%、R² {metric['R²']:.2f}",
    height=650,
)

### 時間序列


In [8]:
all_hours_data = trainer.df_all  # 全部小時（含停機、開停機過渡）

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=all_hours_data.index,
        y=all_hours_data["kW"],
        mode="lines",
        name="逐時實際耗電",
        line=dict(color="#86b6ef", width=2),
    )
)
fig.add_trace(
    go.Scatter(
        x=all_hours_data.index,
        y=all_hours_data["kW_PRED"],
        mode="lines",
        name="逐時預測耗電",
        line=dict(color="#2a78d6", width=2),
    )
)
fig.update_yaxes(title_text=label("kW"))
fig.update_layout(hovermode="x unified")
date_axis(fig)
show(fig, "CH-2 耗電時間序列：預測 vs 實際")

metric_all = trainer.metric_all
print(
    f"全部小時（{all_hours_data.index.min():%m/%d} ~ {all_hours_data.index.max():%m/%d}"
    f"）："
    f"CV(RMSE) = {metric_all['CV(RMSE) %']:.1f}%、R² = {metric_all['R²']:.2f}"
    f"（含開機、停機過渡小時，僅供參考）"
)

全部小時（09/17 ~ 10/13）：CV(RMSE) = 16.5%、R² = 0.99（含開機、停機過渡小時，僅供參考）


In [9]:
# 誤差依小時類型分開看
model_hours = all_hours_data.index.isin(model_data.index)
steady_hours = all_hours_data.index.isin(steady_data.index)
hour_types = [
    ("建模小時", model_hours),
    ("負荷過大", steady_hours & ~model_hours),
    ("開機、停機過渡", all_hours_data["RUNNING"] & ~steady_hours),
]
for type_name, hour_mask in hour_types:
    type_data = all_hours_data.loc[hour_mask, ["kW", "kW_PRED"]].dropna()
    mean_error = (type_data["kW_PRED"] - type_data["kW"]).mean()
    print(
        f"{type_name}：{len(type_data)} 小時，"
        f"平均實際 {type_data['kW'].mean():.0f} kW，平均誤差 {mean_error:+.1f} kW"
    )

建模小時：72 小時，平均實際 173 kW，平均誤差 -0.0 kW
負荷過大：25 小時，平均實際 235 kW，平均誤差 +12.2 kW
開機、停機過渡：67 小時，平均實際 108 kW，平均誤差 -7.7 kW


## 5. 出水溫度調整情境：−1、−0.5、+0.5、+1、+1.5°C

- **節電量**：同一小時用實際與調整後出水溫度（溫差、流量不變）各預測一次相減；調低時為負
- **節電率** = 節電量 ÷ 實際耗電；負荷 > 360 RT 或剛開機的小時不調整
- **95% 區間**：以天為單位重抽樣 500 次重算；調高的下限略低於 0（約 −0.1%~−0.4%），需試行確認
- −1°C 與 +1.5°C 超出過去出水溫度範圍（8.5~9.8°C），可信度較低

下面四張圖：出水溫度、逐時耗電、逐時節電量、逐時節電率（調低藍、調高黃到紅、實際灰）。

In [10]:
# 情境配色：調低用藍、調高由黃到紅，幅度越大越深；實際用灰
SCENARIO_COLORS = {
    -1.0: "#403990",
    -0.5: "#80A6E2",
    0.5: "#FBDD85",
    1.0: "#F46F43",
    1.5: "#CF3D3E",
}
ACTUAL_COLOR = "#898781"

hourly = ch2.hourly_scenario(trainer, DELTAS)
running_hours = hourly["BASE"] > 0
keep_hours = ch2.no_shift(all_hours_data)  # 負荷過大、剛開機：不調整
actual = {
    "T_CHWS": all_hours_data["T_CHWS"].where(all_hours_data["RUNNING"]),
    "kW": all_hours_data["kW"],
}

results = {}
summary_rows = []
for delta in DELTAS:
    saving = hourly["BASE"] - hourly[delta]  # 負值 = 耗電增加
    # 耗電變化 % 的 95% 區間，負值為節電
    change_low, change_high = ch2.bootstrap_pct(trainer, delta)
    # 停機小時節電率記 0，只有缺資料的小時留空
    saving_pct = saving / actual["kW"] * 100
    saving_pct = saving_pct.where(running_hours, 0)
    saving_pct = saving_pct.where(hourly["BASE"].notna())
    results[delta] = {
        "T_CHWS": actual["T_CHWS"] + delta * ~keep_hours,
        "kW": actual["kW"] - saving,
        "SAVING": saving,
        "SAVING_PCT": saving_pct,
    }
    summary_rows.append(
        {
            "出水溫度調整": f"{delta:+.1f}°C",
            "節電 (kWh)": saving[running_hours].sum() * ch2.STEP_HOURS,
            "節電率（資料推估）%": saving[running_hours].sum()
            / actual["kW"][running_hours].sum()
            * 100,
            "節電率（理論估計）%": -ch2.ideal_change_pct(steady_data, delta),
            "95% 區間下限 %": -change_high,
            "95% 區間上限 %": -change_low,
        }
    )

actual_kwh = actual["kW"][running_hours].sum() * ch2.STEP_HOURS
no_shift_count = (running_hours & keep_hours).sum()
print(
    f"有開機 {running_hours.sum()} 小時，實際耗電 {actual_kwh:,.0f} kWh；"
    f"其中 {no_shift_count} 小時負荷過大或剛開機、不調整。節電為負表示耗電增加"
)
pd.DataFrame(summary_rows).set_index("出水溫度調整").round(2)

有開機 164 小時，實際耗電 25,592 kWh；其中 47 小時負荷過大或剛開機、不調整。節電為負表示耗電增加


,節電 (kWh),節電率（資料推估）%,節電率（理論估計）%,95% 區間下限 %,95% 區間上限 %
出水溫度調整,,,,,
-1.0°C,-284.98,-1.11,-2.84,-5.20,0.24
-0.5°C,-142.49,-0.56,-1.42,-2.60,0.12
+0.5°C,142.49,0.56,1.41,-0.12,2.60
+1.0°C,284.98,1.11,2.82,-0.24,5.20
+1.5°C,427.47,1.67,4.22,-0.37,7.80


In [11]:
# 所有情境畫在同一張圖；with_actual 時加入實際值，圖例依溫度由低到高
def plot_scenarios(
    column: str,
    title: str,
    y_title: str,
    with_actual: bool = False,
    y_range: list | None = None,
) -> None:
    traces = []
    for delta in DELTAS:
        traces.append(
            (f"出水溫度 {delta:+.1f}°C", results[delta][column], SCENARIO_COLORS[delta])
        )
    if with_actual:
        # 實際值插在調低情境之後、調高情境之前
        insert_at = sum(delta < 0 for delta in DELTAS)
        traces.insert(insert_at, ("實際", actual[column], ACTUAL_COLOR))

    fig = go.Figure()
    for trace_name, y_values, color in traces:
        fig.add_trace(
            go.Scatter(
                x=hourly.index,
                y=y_values,
                mode="lines",
                name=trace_name,
                line=dict(color=color, width=2),
            )
        )
    if not with_actual:
        # 0 以上節電、以下耗電增加
        fig.add_hline(y=0, line=dict(color=REF_COLOR, width=1))
    fig.update_yaxes(title_text=y_title, range=y_range)
    fig.update_layout(hovermode="x unified")
    date_axis(fig)
    show(fig, title)


# Y 軸下限：最低出水溫度再調低後，取到 0.5°C
T_LOW = np.floor((steady_data["T_CHWS"].min() + min(DELTAS)) * 2) / 2
plot_scenarios(
    "T_CHWS",
    "出水溫度：實際 vs 各情境",
    label("T_CHWS"),
    with_actual=True,
    y_range=[T_LOW, 12],
)

In [12]:
plot_scenarios("kW", "逐時耗電：實際 vs 各情境", label("kW"), with_actual=True)

In [13]:
plot_scenarios("SAVING", "逐時節電量（負值為耗電增加）", "節電量 (kW)")

In [14]:
plot_scenarios("SAVING_PCT", "逐時節電率（節電量 ÷ 實際耗電）", "節電率 (%)")

In [15]:
# 理論驗算：理想冷凍機只看溫度落差，比較調高、調低同幅度時的耗電變化
temperature_lift = ch2.T_COND - (steady_data["T_CHWS"] - ch2.EVAP_DT)
print(
    f"溫度落差（冷凝溫度 {ch2.T_COND:.0f}°C − 蒸發溫度）"
    f"平均約 {temperature_lift.mean():.0f}°C"
)

theory_rows = []
for step in (0.5, 1.0, 1.5):
    saving_up = -ch2.ideal_change_pct(steady_data, step)  # 調高：省電 %
    extra_down = ch2.ideal_change_pct(steady_data, -step)  # 調低：多耗電 %
    theory_rows.append(
        {
            "調整幅度": f"{step:.1f}°C",
            "調高：省電 %": saving_up,
            "調低：多耗電 %": extra_down,
            "兩者相差（佔效果的 %）": (extra_down - saving_up) / saving_up * 100,
        }
    )
pd.DataFrame(theory_rows).set_index("調整幅度").round(2)

溫度落差（冷凝溫度 34°C − 蒸發溫度）平均約 27°C


,調高：省電 %,調低：多耗電 %,兩者相差（佔效果的 %）
調整幅度,,,
0.5°C,1.41,1.42,0.36
1.0°C,2.82,2.84,0.72
1.5°C,4.22,4.27,1.08


## 6. 存模型

存進 `model_registry`（`ch2_power_graybox`，會覆蓋舊模型），需要時再執行。

In [16]:
# await ch2.PowerModelSaver(trainer).save_to_db()

## 7. 結論與限制

**結論**
- 耗電模型 CV(RMSE) 約 3.1%，遠優於 30% 門檻
- 調高出水溫度會節電：每 1°C 約 1.1%（資料推估）～2.8%（理論上限）
- 出水溫度過去幾乎沒變過，實際落在哪裡需試行確認

**限制**
- 只有 09/17 ~ 10/13 資料；−1°C、+1.5°C 為外推
- 360 RT 門檻依實測訂定，沒有出水設定值點位可確認主機何時拉不下水溫
- 「溫差、流量不變」是假設：出水調高後空調箱除濕、供冷能力下降，可能要加大流量或室內濕度上升
- 沒有冷卻水溫點位、不用氣象資料，外氣變化的影響沒有納入；不含泵浦、冷卻水塔